In [35]:
import os
import sys
import io
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"
import pandas as pd
import numpy as np
from tqdm import tqdm
import cv2
from PIL import Image
import torch
import gc
import warnings

source_path = os.path.abspath(os.path.join(os.getcwd(), '..', '..'))
sys.path.append(source_path)

warnings.filterwarnings('ignore')

In [63]:
random_tensor = torch.randn(3, 224, 224)
test_image=generate_cross_image(cross_width_ratio=0.5, cross_height_ratio=0.5)
model_list=global_vars.list_models()
N_max=282
patches=True
pooling=False # if true in transformer mdoels use pooling, if false only the cls token
custom_transform=False
transform_mode='resize'
save_h5=False
truncation = 'remove head'
running = 'new-laptop'
saved = 'old-laptop'
model_mode = 'truncated' #'as is', 'truncated
batch_size = 32
select_cls=False
num_workers=0
normalization=True

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device is: ",device)

Device is:  cuda


In [64]:
df_pre_patch, df_pre_extracted,_,_ =file_IO.preprocess_experiment_logs(source_path)

In [65]:
squares_standard_name = 'squares_gw5.0_m5.0_idx2' #'rectangles_gw3.0_m5.0_idx1' #'squares_gw5.0_m5.0_idx2'
print(file_IO.get_experiment_from_unique_name(df_pre_patch, squares_standard_name))
input_file_name='icdar_train_df_patches_20250515_164130.csv'
viz_numeric=file_IO.get_reference_table_for_experiments(df_pre_patch,df_pre_extracted,im_show=False,im_plot=False)
already_used_models=file_IO.get_models_applied_to_unique_name(viz_numeric, squares_standard_name)
print("Already used models on ",squares_standard_name,": ",already_used_models)

icdar_train_df_patches_20250515_164130.csv
Already used models on  squares_gw5.0_m5.0_idx2 :  ['trocr-small-stage1', 'trocr-small-handwritten', 'resnet50', 'vit-base-patch16-224-in21k', 'trocr-base-handwritten', 'clip-vit-large-patch14', 'trocr-large-handwritten', 'dresnet50', 'crnn_vgg16_bn', 'vitstr_base', 'trocr-large-stage1', 'trocr-base-stage1', 'alexnet', 'vgg16', 'googlenet', 'resnet18', 'DeiT-Tiny']


In [ ]:
for i in range(39,len(model_list)):
    name=model_list[i]
    print(i)
    if name in already_used_models: 
        '''print("Skipping already used model: ",name)
        print('-'*40)
        print('-'*40)
        continue'''
        print("Already used")
    print("Using model: ", name)
    huggingface=global_vars.get_props(name).hugging
    selected_model=name
    transform = u_transforms.get_transform(selected_model, use_patches=patches, custom=custom_transform, mode=transform_mode)
    model = model_utils.get_model(name=selected_model, mode=model_mode, pretrained=True, truncation=truncation)
    torch.cuda.empty_cache()
    gc.collect() 
    model = model.to(device)
    output=compute_output(model, device, transform, huggingface,test_image,show_image=False)
    print("Output shape: ", output.shape)
    n_par,n_layers = count_model_parameters_and_layers(model)
    print('n parameters',f"{n_par:.3e}")
    #print('n layers',n_layers)
    #print(transform)
    print('-'*40)
    print('-'*40)

39
Using model:  alexnet_gap
Output shape:  torch.Size([1, 256])
n parameters 2.470e+06
----------------------------------------
----------------------------------------
40
Using model:  densenet161
Output shape:  torch.Size([1, 2208])
n parameters 2.647e+07
----------------------------------------
----------------------------------------
41
Using model:  densenet121
Output shape:  torch.Size([1, 1024])
n parameters 6.954e+06
----------------------------------------
----------------------------------------
42
Using model:  densenet201
Output shape:  torch.Size([1, 1920])
n parameters 1.809e+07
----------------------------------------
----------------------------------------
43
Using model:  vgg11
Output shape:  torch.Size([1, 512])
n parameters 9.220e+06
----------------------------------------
----------------------------------------
44
Using model:  vgg16_512
Output shape:  torch.Size([1, 512])
n parameters 1.471e+07
----------------------------------------
--------------------------

In [ ]:
print(global_vars.get_props("swin_v2_b"))
print(global_vars.get_props("swin_v2_b").hugging)
print(global_vars.list_models()[:10])

ModelProps(hugging=False, library='torchvision', architecture='transformer')
False
['swin_v2_b', 'swin_v2_s', 'DeiT-Small', 'DeiT-Small-Dist', 'DeiT-Base', 'BEiT-Base', 'BEiT-Large', 'clip-vit-base-patch16', 'clip-vit-base-patch32', 'clip-vit-large-patch14-un']


# reload

In [69]:
import matplotlib.pyplot as plt
def reload_modules():
    import importlib
    import utils.model_utils as model_utils
    import utils.utils_transforms as u_transforms
    import utils.global_vars as global_vars
    import utils.file_IO as file_IO

    importlib.reload(model_utils)
    importlib.reload(u_transforms)
    importlib.reload(global_vars)
    importlib.reload(file_IO)

    return model_utils, u_transforms, global_vars,file_IO
model_utils, u_transforms, global_vars, file_IO = reload_modules()
def generate_cross_image(resolution=(512, 512), cross_width_ratio=0.1, cross_height_ratio=0.1, color=(0, 0, 0)):
    """
    Generates an RGB image with a cross at its center.

    Args:
        resolution (tuple): The resolution of the image (width, height).
        cross_width_ratio (float): The ratio of the cross width to the image width.
        cross_height_ratio (float): The ratio of the cross height to the image height.
        color (tuple): The color of the cross in RGB format (0-255).

    Returns:
        np.ndarray: The generated image as a NumPy array.
    """
    width, height = resolution
    image = np.zeros((height, width, 3), dtype=np.uint8)+255

    cross_width = int(width * cross_width_ratio)
    cross_height = int(height * cross_height_ratio)

    # Draw horizontal bar of the cross
    start_x = (width - cross_width) // 2
    end_x = start_x + cross_width
    start_y_h = (height - int(height * 0.05)) // 2 # Small height for the horizontal bar
    end_y_h = start_y_h + int(height * 0.05)
    cv2.rectangle(image, (start_x, start_y_h), (end_x, end_y_h), color, -1)

    # Draw vertical bar of the cross
    start_x_v = (width - int(width * 0.05)) // 2 # Small width for the vertical bar
    end_x_v = start_x_v + int(width * 0.05)
    start_y = (height - cross_height) // 2
    end_y = start_y + cross_height
    cv2.rectangle(image, (start_x_v, start_y), (end_x_v, end_y), color, -1)

    return image
def compute_output(model, device, transform, huggingface,test_image, show_image=True):
    model.eval()
    if isinstance(test_image, np.ndarray):
      patch=Image.fromarray(test_image)
    else:
      patch = test_image
    if huggingface:
        # the transform is actually an huggingface processor in this case
        inputs = transform(images=patch, return_tensors="pt")
        # Remove batch dimension from inputs
        patch = inputs['pixel_values'].squeeze()
    else:
        patch = transform(patch)

    if show_image:
      img_np = patch.permute(1, 2, 0).cpu().numpy()
      # Normalize if needed
      print("min,max",img_np.min(),img_np.max())
      img_np = (img_np - img_np.min()) / (img_np.max() - img_np.min())

      plt.subplot(1, 2, 2)
      plt.imshow(img_np)
      plt.title("Transformed")
      plt.axis('off')
      plt.show()

    patch = patch.to(device)
    output = model(patch.unsqueeze(0))
    return output
# Example usage:
# image = generate_cross_image()
# Image.fromarray(image).save("cross_image.png")
def count_model_parameters_and_layers(model):
    """
    Counts the number of parameters and layers in a PyTorch model.

    Args:
        model (torch.nn.Module): The PyTorch model.

    Returns:
        tuple: A tuple containing the total number of parameters and the number of layers.
    """
    num_parameters = sum(p.numel() for p in model.parameters())
    num_layers = len(list(model.children()))
    return num_parameters, num_layers